In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

# =========================================
# helper function: integer -> binary vector
# =========================================
def int_to_bin(x, bits):
    return torch.tensor(
        [int(b) for b in format(x, f'0{bits}b')],
        dtype=torch.float32
    )

# =========================================
# prepare dataset for binary addition
# =========================================
BITS = 4
MAX_NUMBER = 2 ** BITS

X = []
Y_add = []
Y_mul2 = []

for a in range(MAX_NUMBER):
    for b in range(MAX_NUMBER):
        a_bin = int_to_bin(a, BITS)
        b_bin = int_to_bin(b, BITS)

        # input = concatenation of a and b
        X.append(torch.cat((a_bin, b_bin)))

        # output for addition: a + b, needs BITS+1 bits
        Y_add.append(int_to_bin(a + b, BITS + 1))

# convert lists to tensors
X = torch.stack(X)          # shape: (256, 8)
Y_add = torch.stack(Y_add)  # shape: (256, 5)

# =========================================
# define model
# =========================================
class BinaryOperationNet(nn.Module):
    def __init__(self, input_size, output_size):
        super(BinaryOperationNet, self).__init__()
        self.hidden1 = nn.Linear(input_size, 32)
        self.hidden2 = nn.Linear(32, 16)
        self.output = nn.Linear(16, output_size)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.hidden1(x)
        x = self.relu(x)
        x = self.hidden2(x)
        x = self.relu(x)
        x = self.output(x)
        return self.sigmoid(x)

# =========================================
# define model and learning scheme
# =========================================
model = BinaryOperationNet(input_size=2 * BITS, output_size=BITS + 1)
optimizer = optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

# =========================================
# train
# =========================================
N_epochs = 5000

train_loss = []
model.train()

for epoch in range(N_epochs):
    optimizer.zero_grad()
    y_pred = model(X)
    loss = loss_fn(y_pred, Y_add)
    loss.backward()
    optimizer.step()
    train_loss.append(loss.item())

    if epoch % 500 == 0:
        print(f"epoch = {epoch}, loss = {loss.item():.6f}")

print("last loss value =", train_loss[-1])

# =========================================
# test a few examples
# =========================================
model.eval()

test_cases = [(3, 5), (7, 6), (8, 7), (15, 15)]

with torch.no_grad():
    for a, b in test_cases:
        x_test = torch.cat((int_to_bin(a, BITS), int_to_bin(b, BITS))).unsqueeze(0)
        y_pred = model(x_test)
        y_pred_rounded = torch.round(y_pred).squeeze(0)

        print(f"{a} + {b}")
        print("input a     =", int_to_bin(a, BITS).tolist())
        print("input b     =", int_to_bin(b, BITS).tolist())
        print("predicted   =", y_pred_rounded.tolist())
        print("true result =", int_to_bin(a + b, BITS + 1).tolist())
        print()

epoch = 0, loss = 0.251573
epoch = 500, loss = 0.005848
epoch = 1000, loss = 0.004789
epoch = 1500, loss = 0.004726
epoch = 2000, loss = 0.004708
epoch = 2500, loss = 0.004699
epoch = 3000, loss = 0.004695
epoch = 3500, loss = 0.004692
epoch = 4000, loss = 0.004691
epoch = 4500, loss = 0.004690
last loss value = 0.00468913558870554
3 + 5
input a     = [0.0, 0.0, 1.0, 1.0]
input b     = [0.0, 1.0, 0.0, 1.0]
predicted   = [0.0, 1.0, 0.0, 0.0, 0.0]
true result = [0.0, 1.0, 0.0, 0.0, 0.0]

7 + 6
input a     = [0.0, 1.0, 1.0, 1.0]
input b     = [0.0, 1.0, 1.0, 0.0]
predicted   = [0.0, 1.0, 1.0, 0.0, 1.0]
true result = [0.0, 1.0, 1.0, 0.0, 1.0]

8 + 7
input a     = [1.0, 0.0, 0.0, 0.0]
input b     = [0.0, 1.0, 1.0, 1.0]
predicted   = [0.0, 1.0, 1.0, 1.0, 1.0]
true result = [0.0, 1.0, 1.0, 1.0, 1.0]

15 + 15
input a     = [1.0, 1.0, 1.0, 1.0]
input b     = [1.0, 1.0, 1.0, 1.0]
predicted   = [1.0, 1.0, 1.0, 1.0, 0.0]
true result = [1.0, 1.0, 1.0, 1.0, 0.0]



In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

# =========================================
# helper function: integer -> binary vector
# =========================================
def int_to_bin(x, bits):
    return torch.tensor([int(b) for b in format(x, f'0{bits}b')], dtype=torch.float32)

# =========================================
# parameters
# =========================================
BITS = 4
MAX_NUMBER = 2 ** BITS

# =========================================
# dataset for ADD
# input = [a_bits, b_bits]
# output = binary(a+b)
# =========================================
X_add = []
Y_add = []

for a in range(MAX_NUMBER):
    for b in range(MAX_NUMBER):
        X_add.append(torch.cat((int_to_bin(a, BITS), int_to_bin(b, BITS))))
        Y_add.append(int_to_bin(a + b, BITS + 1))

X_add = torch.stack(X_add)
Y_add = torch.stack(Y_add)

# =========================================
# dataset for MUL2
# input = [a_bits, 0,0,0,0]  -> same input size as ADD
# output = binary(2*a)
# =========================================
X_mul2 = []
Y_mul2 = []

zero_bin = torch.zeros(BITS)

for a in range(MAX_NUMBER):
    X_mul2.append(torch.cat((int_to_bin(a, BITS), zero_bin)))
    Y_mul2.append(int_to_bin(2 * a, BITS + 1))

X_mul2 = torch.stack(X_mul2)
Y_mul2 = torch.stack(Y_mul2)

operations = ("ADD", "MUL2")
datasets = ((X_add, Y_add), (X_mul2, Y_mul2))

# =========================================
# define model
# =========================================
class BinaryNet(nn.Module):
    def __init__(self, input_size, output_size):
        super(BinaryNet, self).__init__()
        self.hidden1 = nn.Linear(input_size, 32)
        self.hidden2 = nn.Linear(32, 16)
        self.output = nn.Linear(16, output_size)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.hidden1(x)
        x = self.relu(x)
        x = self.hidden2(x)
        x = self.relu(x)
        x = self.output(x)
        return self.sigmoid(x)

# =========================================
# training
# =========================================
N_epochs = 5000

for operation, (X_data, Y_data) in zip(operations, datasets):
    print("=" * 50)
    print(f"OPERATION: {operation}")
    print("=" * 50)

    model = BinaryNet(input_size=2 * BITS, output_size=BITS + 1)
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    MSE = nn.MSELoss()

    train_loss = []
    model.train()

    for epoch in range(N_epochs):
        optimizer.zero_grad()
        y_pred = model(X_data)
        loss = MSE(y_pred, Y_data)
        loss.backward()
        optimizer.step()
        train_loss.append(loss.item())

        if epoch % 500 == 0:
            print(f"epoch = {epoch}, loss = {loss.item():.6f}")

    print(f"last loss value = {train_loss[-1]}")
    print()

    # =========================================
    # test section with styled output
    # =========================================
    model.eval()
    with torch.no_grad():
        if operation == "ADD":
            test_cases = [(3, 5), (7, 6), (8, 7), (15, 15)]

            for a, b in test_cases:
                input_a = int_to_bin(a, BITS)
                input_b = int_to_bin(b, BITS)

                x_test = torch.cat((input_a, input_b)).unsqueeze(0)
                y_pred = model(x_test)
                y_pred_rounded = torch.round(y_pred).squeeze(0)
                y_true = int_to_bin(a + b, BITS + 1)

                print(f"{a} + {b}")
                print(f"input a     = {input_a.tolist()}")
                print(f"input b     = {input_b.tolist()}")
                print(f"predicted   = {y_pred_rounded.tolist()}")
                print(f"true result = {y_true.tolist()}")
                print()

        elif operation == "MUL2":
            test_cases = [3, 5, 7, 11, 15]

            for a in test_cases:
                input_a = int_to_bin(a, BITS)

                x_test = torch.cat((input_a, zero_bin)).unsqueeze(0)
                y_pred = model(x_test)
                y_pred_rounded = torch.round(y_pred).squeeze(0)
                y_true = int_to_bin(2 * a, BITS + 1)

                print(f"{a} * 2")
                print(f"input a     = {input_a.tolist()}")
                print(f"predicted   = {y_pred_rounded.tolist()}")
                print(f"true result = {y_true.tolist()}")
                print()

OPERATION: ADD
epoch = 0, loss = 0.253010
epoch = 500, loss = 0.012599
epoch = 1000, loss = 0.012524
epoch = 1500, loss = 0.012510
epoch = 2000, loss = 0.012506
epoch = 2500, loss = 0.012503
epoch = 3000, loss = 0.012502
epoch = 3500, loss = 0.012501
epoch = 4000, loss = 0.012501
epoch = 4500, loss = 0.012501
last loss value = 0.01250048540532589

3 + 5
input a     = [0.0, 0.0, 1.0, 1.0]
input b     = [0.0, 1.0, 0.0, 1.0]
predicted   = [0.0, 1.0, 0.0, 0.0, 0.0]
true result = [0.0, 1.0, 0.0, 0.0, 0.0]

7 + 6
input a     = [0.0, 1.0, 1.0, 1.0]
input b     = [0.0, 1.0, 1.0, 0.0]
predicted   = [0.0, 1.0, 1.0, 0.0, 1.0]
true result = [0.0, 1.0, 1.0, 0.0, 1.0]

8 + 7
input a     = [1.0, 0.0, 0.0, 0.0]
input b     = [0.0, 1.0, 1.0, 1.0]
predicted   = [0.0, 1.0, 1.0, 1.0, 1.0]
true result = [0.0, 1.0, 1.0, 1.0, 1.0]

15 + 15
input a     = [1.0, 1.0, 1.0, 1.0]
input b     = [1.0, 1.0, 1.0, 1.0]
predicted   = [1.0, 1.0, 1.0, 1.0, 0.0]
true result = [1.0, 1.0, 1.0, 1.0, 0.0]

OPERATION: MUL2
epoc